# Error Handling: The Basics

Understanding how to interpret, use, and raise errors within your code is an important part of writing a robust and user-friendly codebase.
Note that "error handling" is a broad term that encompasses all three; you need to understand how to read an error traceback and error message before you start writing your own code that throws exceptions.

When the interpreter encounters a `raise` command, or encounters a problem when performing an operation when running a portion of code, it will output a **stacktrace** as well as any information that it has (or has been told to provide) concerning the problem.
Exceptions are raised when the program has encountered a problem that it cannot overcome, and needs to terminate early.

The majority of errors that you'll encounter will likely be from core- or third-party libraries that you are using.
The `raise` keyword is what you use to throw errors when writing your own code - more on why you'd want to do this later.

## Refresher: Stacktraces and `raise`

For now, let's quickly go over how to read a stacktrace.
In the code below, we deliberately try to access an element of a list that does not exist.
The reason why we're wrapping this inside a function will become clear when we examine the result of running the code block.

In [4]:
def add_elements_pairwise(my_list: list) -> list:
    """Add elements pairwise, returning the result."""
    even_indexes = my_list[::2]
    odd_indexes = my_list[1::2]

    return [
        even + odd_indexes[i] for i, even in enumerate(even_indexes)
    ]

list_of_6 = list(range(6))
list_of_7 = list(range(7))

print("List of 6 added pairwise:", add_elements_pairwise(list_of_6))
print("List of 7 added pairwise:", add_elements_pairwise(list_of_7))

List of 6 added pairwise: [1, 5, 9]


IndexError: list index out of range

The stacktrace above provides a lot (sometimes too much!) information about a problem that we encountered.

Typically, you want to start reading stacktraces **from the bottom**: 99% of the time there will be enough information here for you to figure out what the issue is.

- At the very bottom, we have the exception type that was raised, and the error message itself.
  In this case, Python has thrown an `IndexError` exception, and the information (message) about why we received this error is `"list index out of range"`.
- Above the exception information, Python then prints out the stacktrace in blocks.
  This essentially highlights the line of code that Python was running when the exception was encountered.
  These blocks are nested from innermost (at the bottom) to outermost (at the top).
  In this case; we can see that the exception was encountered on line 7, whilst inside the list comprehension that started on line 6, whilst running line 14 of the cell.

At this point, you might use a debugger to determine the cause of the exception. If the error message is descriptive enough you may be able to understand the cause without using a debugger.
In this case, the `IndexError` and error message indicate to us that we have tried to access an index of a list that does not exist.
Indeed, this is because when we run `add_elements_pairwise(list_of_7)`, `even_indexes` is a list of 4 elements whilst `odd_indexes` is a list of 3 elements.
As such, when we do `for i, even in enumerate(even_indexes)`, we will eventually hit index `i = 3`.
Then, when we attempt to access `odd_indexes[i]` when `i=3`, Python realises there is no element at this index in `odd_indexes`, and so throws the above exception to terminate the program (and tell us _why_ it was terminated).

Note that the above error is thrown by one of Python's internal libraries.
The `raise` keyword exists to allow you to throw your own exceptions should the circumstances arise:

In [ ]:
def add_elements_pairwise(my_list: list) -> list:
    """Add elements pairwise, returning the result."""
    even_indexes = my_list[::2]
    odd_indexes = my_list[1::2]

    if len(odd_indexes) != len(even_indexes):
        raise ValueError("Error message text goes here")

    return [even + odd_indexes[i] for i, even in enumerate(even_indexes)]


list_of_6 = list(range(6))
list_of_7 = list(range(7))

print("List of 6 added pairwise:", add_elements_pairwise(list_of_6))
print("List of 7 added pairwise:", add_elements_pairwise(list_of_7))

List of 6 added pairwise: [1, 5, 9]


ValueError: Error message text goes here

Notice how the stacktrace has changed now that we have updated our code.
This is because Python now encounters our `raise` statement within `add_elements_pairwise`, and the `Exception` we provide to the `raise` keyword is then thrown to the user.

Indeed, the `ValueError` (and it's accompanying message) we provided now appears as the "reason" for Python's early termination.

## Why Use Exceptions?

The purpose of raising an error is to halt the execution of a program early due to a problem, whilst _also_ providing information to the user on why the problem occurred and suggestions for overcoming it.

Note that `raise`-ing an exception is different to forcing termination of the program, via something like `quit()` or `exit()` (which in general you should never use in your codebase!).

- `exit()` and `quit()` actually raise the `SystemExit` exception! This will kill the Python process, but is considered "dirty".
  In particular, if you do encounter a problem and resolve it by calling `quit`/`exit` as opposed to raising a proper exception, you'll be losing all information about the actual problem you encountered.
- `sys.exit()` is considered the cleanest way to exit Python.
  You can even provide an exit code as an argument to indicate to the system whether Python exited successfully or not.
  However, this has the same problems as `quit`/`exit` when using it when you encounter a problem.
- `raise Exception(msg)` ensures that the user gets a stacktrace to examine, as well as some context as to what caused the problem (the `Exception` type and `msg`).
  It will then also handling proper termination of Python itself, like the other commands above.

As such, the purpose of exceptions is primarily to provide _information_.
They are a succinct way of giving context and information about a problem to the user.

## Informative Exceptions

As the primary purpose of exceptions is to provide the user with information, it is important to provide relevant information about a problem that causes an exception, within the exception itself.
Exceptions generally do this in two parts:


[spotted in the wild](https://github.com/NSs-FLF-RHUL/QSpin/pull/68#discussion_r3520983969)


### Exception Types

`Exception` is the "base class" for all exceptions in Python, however in general you'll never see someone `raise`-ing a raw `Exception`.
Instead, you'll typically see people raising one of the derived `Exception` subclasses: [a list of which can be found here](https://www.w3schools.com/python/python_ref_exceptions.asp).

Each `Exception` type corresponds to a general problem that might be encountered.
For example,

- `KeyError`s cover cases when you attempt to access a container using a key that it doesn't have.
  This most commonly occurs when working with `dict`s.
- `IndexError`s are typically `raise`d when someone attempts to access elements of containers that don't exist.
- `RuntimeError` is the catch-all for problems that don't fit anywhere else.
- `ValueError`s handle when you attempt to perform an operation using an invalid value.
  For example, trying to add a `float` to a `list`.

When you want to `raise` an exception you should consider which exception type to use.
This is important, as it gives your user an immediate indication of what went wrong, which will likely impact how they go about fixing the problem.

### Error Messages

`Exception`s also contain a message, which is the text that appears at the bottom of the stacktrace when they are raised.

Whilst the exception's type indicates the general, well, type of problem that was encountered, the error message contains the specifics.
As such, error messages should be informative and to the point.

Our example above, where we `ValueError("Error message text goes here")` is an example of a _bad_ error message.
As we can see from the stacktrace, the user only gets the information "error message text does here" in the output - that doesn't tell me anything about what went wrong!

A less egregious example spotted in the DeepSKA wilds [can be seen here](https://github.com/NSs-FLF-RHUL/QSpin/pull/68#discussion_r3520983969) (in fairness to the author of this PR, they did rectify this afterwards).
This is an OK error message, because it at least describes the problem that was encountered.
However, it doesn't actually give any context, which is what the suggested fix is doing - rather than just saying "this value is not supported", it is much better to say "you tried to use this unsupported value, but your options are actually these...".
Or if you don't want to list the valid options, direct the user to the location they can find more information about the valid options.

To fix our example above, something like the following would be appropriate:

In [5]:
def add_elements_pairwise(my_list: list) -> list:
    """Add elements pairwise, returning the result."""
    even_indexes = my_list[::2]
    odd_indexes = my_list[1::2]

    if len(odd_indexes) != len(even_indexes):
        raise ValueError(
            "There are different numbers of even-indexed and odd-indexed values. "
            "Pairwise addition cannot be performed when there is an extra element "
            "at the end of the list."
        )

    return [even + odd_indexes[i] for i, even in enumerate(even_indexes)]


list_of_6 = list(range(6))
list_of_7 = list(range(7))

print("List of 6 added pairwise:", add_elements_pairwise(list_of_6))
print("List of 7 added pairwise:", add_elements_pairwise(list_of_7))

List of 6 added pairwise: [1, 5, 9]


ValueError: There are different numbers of even-indexed and odd-indexed values. Pairwise addition cannot be performed when there is an extra element at the end of the list.